In [1]:
# Backtesting Strategy
# This moves up a folder to access the juicy .py data
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import data as dt
import models as md
import backtest as bt
"""
Dynamic Airline Factor Laboratory
_______
Trading engine separated into two conceptual stages following the mentor's
advice:

    STAGE 1 — PORTFOLIO CONSTRUCTION
        Converts z-scores into directional signals and dollar-neutral weights.
        Functions: generate_signals() and construct_portfolio()

    STAGE 2 — EXECUTION & PERFORMANCE
        Applies transaction costs, computes PnL, evaluates performance
        including benchmark regression against JETS and SPY.
        Functions: compute_portfolio_returns(), compute_performance(),
                   benchmark_regression(), macro_regime_diagnostics()

Strategy Logic
_______
Monthly rebalancing. At each month-end date, using z-scores computed from
residuals estimated with the look-ahead-clean covariance matrix:

    Entry Long  : z < -2.0  (airline underperformed factor model → buy)
    Entry Short : z > +2.0  (airline outperformed factor model → sell)
    Hold        : -2 ≤ z ≤ -0.5  or  +0.5 ≤ z ≤ +2  (within bounds)
    Exit        : |z| < 0.5  (mispricing has reverted)

Position sizing: equal-weight within longs and shorts, dollar-neutral.
Execution: positions formed at end of day t, executed at open of day t+1.
           (Implemented via a 1-day shift on the position DataFrame.)
Transaction costs: 15 basis points one-way on turnover.
"""

"\nDynamic Airline Factor Laboratory\n_______\nTrading engine separated into two conceptual stages following the mentor's\nadvice:\n\n    STAGE 1 — PORTFOLIO CONSTRUCTION\n        Converts z-scores into directional signals and dollar-neutral weights.\n        Functions: generate_signals() and construct_portfolio()\n\n    STAGE 2 — EXECUTION & PERFORMANCE\n        Applies transaction costs, computes PnL, evaluates performance\n        including benchmark regression against JETS and SPY.\n        Functions: compute_portfolio_returns(), compute_performance(),\n                   benchmark_regression(), macro_regime_diagnostics()\n\nStrategy Logic\n_______\nMonthly rebalancing. At each month-end date, using z-scores computed from\nresiduals estimated with the look-ahead-clean covariance matrix:\n\n    Entry Long  : z < -2.0  (airline underperformed factor model → buy)\n    Entry Short : z > +2.0  (airline outperformed factor model → sell)\n    Hold        : -2 ≤ z ≤ -0.5  or  +0.5 ≤ z ≤ +2

In [2]:
airline_returns, macro_returns, prices = dt.load_data()

Airlines : 6 tickers
Macro : 4 tickers
Dates : 2903 trading days
Range: 2015-01-05 -> 2026-07-16
Training : 2015-01-05 -> 2019-12-31
Validation: 2022-01-01 -> 2026-07-16


In [3]:
covs = md.rolling_covariances(airline_returns)
systematic, residuals, eigvals, eigvecs, explained = \
    md.compute_factor_model(
        airline_returns,
        covs
    )
zscores = md.compute_rolling_zscore(residuals)
zscores

NameError: name 'LedoitWolf' is not defined

In [ ]:
signals = bt.generate_signals(zscores)
signals.head()

In [ ]:
signals.sum(axis=1).plot(figsize=(12,5))
plt.title("Net Trading Signals")
plt.ylabel("Net Positions")
plt.show()

In [ ]:
positions = bt.construct_portfolio(signals)
positions.head()

In [ ]:
net_returns, trades = bt.compute_portfolio_returns(
    positions,
    airline_returns
)

In [ ]:
performance = bt.compute_performance(
    net_returns,
    trades,
    macro_returns
)
performance

In [ ]:
equity = (1 + net_returns).cumprod()
equity.plot(figsize=(14,6))
plt.title("Strategy Equity Curve")
plt.ylabel("Growth of $1")
plt.grid()
plt.show()

In [ ]:
rolling_max = equity.cummax()
drawdown = (equity - rolling_max) / rolling_max
drawdown.plot(figsize=(14,4))
plt.title("Strategy Drawdown")
plt.show()

In [ ]:
rolling_sharpe = (
    net_returns.rolling(252).mean()
    /
    net_returns.rolling(252).std()
) * np.sqrt(252)
rolling_sharpe.plot(figsize=(14,5))
plt.title("Rolling 1-Year Sharpe")
plt.show()

In [ ]:
trades["turnover"].plot(figsize=(12,4))
plt.title("Daily Portfolio Turnover")
plt.show()

In [ ]:
trades["n_longs"].plot(label="Longs")
trades["n_shorts"].plot(label="Shorts")
plt.legend()
plt.title("Number of Active Positions")
plt.show()

In [ ]:
performance[
    [
        "jets_alpha_annual",
        "jets_beta",
        "jets_r2",
        "spy_alpha_annual",
        "spy_beta",
        "spy_r2",
    ]
]

In [ ]:
regimes = bt.macro_regime_diagnostics(
    net_returns,
    macro_returns
)
regimes

In [ ]:
signals.head(20)
positions.head(20)

In [ ]:
performance.to_csv("performance.csv")

trades.to_csv("trades.csv")

net_returns.to_csv("net_returns.csv")